# Site inference

Generate a CSV with one model score per RNA site. The default uses the **saved H04 model** from run
`20261005T103445_98fad904` and the **18,386-site internal test split** behind the reported metrics.

1. Select the repository's `.venv` Python kernel.
2. Edit the **Settings** cell to choose the model and dataset.
3. Use **Restart Kernel and Run All**. The last cells show the output path and highest-scoring sites.

This notebook loads an already fitted model; it does not train or tune it. Each run saves a new CSV and
an accompanying `.metadata.json` recording the model and dataset used.

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json
import sys
import uuid

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src/inference.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository root or notebooks/ folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.inference import load_model_bundle, predict_to_csv

print("Project:", ROOT)

Project: /Users/jiacheng/Documents/GitHub/DSA4262_Genomies


## Settings: change the model or dataset here

- **`MODEL_PATH`**: path to a compatible saved `selected_model.joblib`. Different model families are
  supported when saved by `data0_modelling_pipeline.ipynb`; their saved feature selection is reused.
  The separate `newadjustmentfolder` bundles have a different format and are not supported here.
- **`DATASET`**: `"internal_test"`, `"data1"`, `"data2"`, or `"custom"`.
- For a custom dataset, set **`CUSTOM_INPUT_PATH`** and **`CUSTOM_INPUT_KIND`**:
  - `"reads"`: project JSON/JSONL, optionally gzip-compressed, or parsed per-read Parquet.
  - `"features"`: a site-level CSV/Parquet containing site identifiers and the model's feature columns.
    A CSV containing only previously predicted scores is not a feature dataset.
- **`OUTPUT_DIR`**: where new prediction files go. Filenames include the model, dataset, and run time.

`INTERNAL_TEST_DIR` chooses the test dataset independently of `MODEL_PATH`. Changing the model does not
silently change the test split. Changing to another model requires a compatible feature schema.

In [2]:
# ----- EDIT THESE SETTINGS -----
H04_RUN_DIR = ROOT / "data/processed/modelling/data0_gene_split_seed42/person_c_runs/20261005T103445_98fad904"
MODEL_PATH = H04_RUN_DIR / "selected_model.joblib"

DATASET = "internal_test"  # "internal_test", "data1", "data2", or "custom"
INTERNAL_TEST_DIR = H04_RUN_DIR

CUSTOM_INPUT_PATH = ROOT / "data/raw/data1/dataset1.json.gz"
CUSTOM_INPUT_KIND = "reads"  # "reads" or "features"

OUTPUT_DIR = ROOT / "data/processed/inference"
BATCH_SIZE = 200_000  # Read rows (reads mode) or site rows (features mode)
MODEL_THREADS = 4
# ----- END SETTINGS -----

DATASETS = {
    "internal_test": (INTERNAL_TEST_DIR / "X_test.parquet", "features"),
    "data1": (ROOT / "data/raw/data1/dataset1.json.gz", "reads"),
    "data2": (ROOT / "data/raw/data2/dataset2.json.gz", "reads"),
    "custom": (CUSTOM_INPUT_PATH, CUSTOM_INPUT_KIND),
}
if DATASET not in DATASETS:
    raise ValueError(f"Choose DATASET from {list(DATASETS)}")
INPUT_PATH, INPUT_KIND = DATASETS[DATASET]

print("Model:", MODEL_PATH)
print("Dataset:", DATASET)
print("Input:", INPUT_PATH)
print("Input kind:", INPUT_KIND)

Model: /Users/jiacheng/Documents/GitHub/DSA4262_Genomies/data/processed/modelling/data0_gene_split_seed42/person_c_runs/20261005T103445_98fad904/selected_model.joblib
Dataset: internal_test
Input: /Users/jiacheng/Documents/GitHub/DSA4262_Genomies/data/processed/modelling/data0_gene_split_seed42/person_c_runs/20261005T103445_98fad904/X_test.parquet
Input kind: features


## Confirm the saved model

The default should say **H04: hgb (engineered, none)**. Its validation cutoff is about **0.186744**.
The cutoff is not needed to generate scores. Load only saved model files that you trust.

In [3]:
bundle = load_model_bundle(MODEL_PATH)
print("Model name:", bundle.get("model_name", type(bundle["model"]).__name__))
print("Training run:", bundle.get("run_id", "not recorded"))
print("Validation threshold:", bundle.get("threshold", "not recorded"))
print("Input feature columns:", len(bundle["feature_columns"]))
print("Model parameters:", bundle.get("experiment", {}).get("params", "see fitted model"))

Model name: H04: hgb (engineered, none)
Training run: 20261005T103445_98fad904
Validation threshold: 0.18674391737784252
Input feature columns: 162
Model parameters: {'min_samples_leaf': 20, 'max_leaf_nodes': 31, 'max_iter': 350, 'learning_rate': 0.03, 'l2_regularization': 2.0}


## Generate the scores

Raw reads are processed in batches while keeping every site's reads together. Feature calculations
reuse the training code, and feature names are checked against the saved model before prediction.
Large raw datasets take longer than the already prepared internal test features.

In [4]:
model_tag = str(bundle.get("experiment", {}).get("experiment_id", "model"))
model_tag = "".join(c if c.isalnum() or c in "-_" else "_" for c in model_tag)
run_stamp = datetime.now(ZoneInfo("Asia/Singapore")).strftime("%Y%m%dT%H%M%S")
OUTPUT_PATH = OUTPUT_DIR / f"{model_tag}_{DATASET}_{run_stamp}_{uuid.uuid4().hex[:8]}.csv"

run_metadata = predict_to_csv(
    MODEL_PATH, INPUT_PATH, OUTPUT_PATH,
    input_kind=INPUT_KIND, batch_size=BATCH_SIZE, threads=MODEL_THREADS,
)

Model: H04: hgb (engineered, none)
Input: /Users/jiacheng/Documents/GitHub/DSA4262_Genomies/data/processed/modelling/data0_gene_split_seed42/person_c_runs/20261005T103445_98fad904/X_test.parquet (features)
Scored 18,386 sites
Saved 18,386 site scores to /Users/jiacheng/Documents/GitHub/DSA4262_Genomies/data/processed/inference/H04_internal_test_20261005T190048_3e3051ea.csv


## View and locate the CSV

The CSV has exactly three columns: **`transcript_id`**, **`transcript_position`**, and **`score`**.
Scores range from 0 to 1; larger scores indicate stronger model evidence for modification.
They are not guaranteed to be calibrated probabilities. Sorting below only changes the preview;
site identifiers remain attached to every score in the saved file.

In [5]:
predictions = pd.read_csv(OUTPUT_PATH)
print(f"CSV: {OUTPUT_PATH}")
print(f"Model/dataset record: {OUTPUT_PATH.with_suffix('.metadata.json')}")
print(f"Sites scored: {len(predictions):,}")
print(f"Score range: {predictions.score.min():.6f} to {predictions.score.max():.6f}")
display(predictions.head(10))
print("Top 10 sites by model score:")
display(predictions.nlargest(10, "score"))

CSV: /Users/jiacheng/Documents/GitHub/DSA4262_Genomies/data/processed/inference/H04_internal_test_20261005T190048_3e3051ea.csv
Model/dataset record: /Users/jiacheng/Documents/GitHub/DSA4262_Genomies/data/processed/inference/H04_internal_test_20261005T190048_3e3051ea.metadata.json
Sites scored: 18,386
Score range: 0.000405 to 0.896259
Top 10 sites by model score:


,transcript_id,transcript_position,score
0,ENST00000001008,190,0.003373
1,ENST00000001008,257,0.017642
2,ENST00000001008,270,0.006792
3,ENST00000001008,338,0.027375
4,ENST00000001008,389,0.014742
5,ENST00000001008,410,0.023350
6,ENST00000001008,425,0.004914
7,ENST00000001008,458,0.044191
8,ENST00000001008,510,0.002777
9,ENST00000001008,652,0.005250


,transcript_id,transcript_position,score
2566,ENST00000261489,890,0.896259
2040,ENST00000254322,1151,0.895723
1280,ENST00000244020,1147,0.889916
8028,ENST00000339276,952,0.882363
11000,ENST00000371785,1050,0.878387
2054,ENST00000254322,1893,0.878189
2039,ENST00000254322,1124,0.876819
1174,ENST00000241600,921,0.871620
10913,ENST00000371269,2478,0.870802
53,ENST00000007264,1008,0.870147


## Check against the original H04 results

When the saved model and input are the original H04/internal-test pair, this checks every score
against the historical predictions. It is a reproducibility check, not a new performance estimate.
The check is skipped when you choose another model or dataset.

In [6]:
original_pair = (
    Path(MODEL_PATH).resolve() == (H04_RUN_DIR / "selected_model.joblib").resolve()
    and Path(INPUT_PATH).resolve() == (H04_RUN_DIR / "X_test.parquet").resolve()
    and INPUT_KIND == "features"
)
if original_pair:
    keys = ["transcript_id", "transcript_position"]
    historical = pd.read_parquet(H04_RUN_DIR / "test_predictions.parquet")["score"].sort_index()
    current = predictions.set_index(keys)["score"].sort_index()
    pd.testing.assert_index_equal(current.index, historical.index)
    np.testing.assert_allclose(current.to_numpy(), historical.to_numpy(), rtol=1e-12, atol=1e-12)
    print(f"PASS: all {len(current):,} scores match the original H04 test predictions.")
else:
    print("Historical comparison skipped for this model/dataset choice.")

PASS: all 18,386 scores match the original H04 test predictions.
